In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

In [ ]:
train.head()

In [ ]:
train.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
y=np.log1p(train['SalePrice'].values)
X=train.drop(['Id','SalePrice'],axis=1)
X_test=test.drop(['Id'],axis=1)

X=X.fillna('None')
X_test=X_test.fillna('None')

for col in X.select_dtypes(include=['object']).columns:
    lbl = LabelEncoder()
    lbl.fit(list(X[col].astype(str)) + list(X_test[col].astype(str)))
    X[col] = lbl.transform(list(X[col].astype(str)))
    X_test[col] = lbl.transform(list(X_test[col].astype(str)))

X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)

X = X.values
X_test = X_test.values

In [ ]:
models = {
    "Lasso": Lasso(alpha=0.0005, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=0),
    "LightGBM": LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42),
    "Ridge": Ridge(alpha=10, random_state=42),
    "CatBoost": CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_state=42, verbose=0)

}

def get_oof_predictions(model, X, y, X_test, n_splits=5, eval_metric=None):
    from sklearn.base import clone
    from sklearn.metrics import accuracy_score
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    # 自适应输出维度
    n_classes = len(np.unique(y))
    output_shape = (X.shape[0], n_classes) if n_classes > 2 else (X.shape[0], 1)
    test_shape = (n_splits, X_test.shape[0], n_classes) if n_classes > 2 else (n_splits, X_test.shape[0])
    
    oof_train = np.zeros(output_shape)
    oof_test_skf = np.empty(test_shape)
    scores = []
    
    for i, (train_idx, val_idx) in enumerate(kf.split(X)):
        model_clone = clone(model)
        model_clone.fit(X[train_idx], y[train_idx])
        
        # 训练集OOF预测
        oof_train[val_idx] = model_clone.predict(X[val_idx]).reshape(-1, n_classes if n_classes > 2 else 1)
        
        # 测试集预测
        test_pred = model_clone.predict(X_test)
        oof_test_skf[i] = test_pred.reshape(-1, n_classes if n_classes > 2 else 1)
        
        # 评估
        if eval_metric:
            score = eval_metric(y[val_idx], oof_train[val_idx])
            scores.append(score)
            print(f"Fold {i+1} {eval_metric.__name__}: {score:.4f}")
    
    # 测试集预测平均
    oof_test = oof_test_skf.mean(axis=0)
    
    if scores:
        print(f"OOF Mean {eval_metric.__name__}: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")
    
    return oof_train, oof_test

In [ ]:
oof_train_list = []
oof_test_list = []
cv_scores = {}

for name, model in models.items():
    try:
        oof_train, oof_test = get_oof_predictions(model, X, y, X_test)
        score = np.sqrt(mean_squared_error(y, oof_train.ravel()))
        cv_scores[name] = score
        
        oof_train_list.append(oof_train)
        oof_test_list.append(oof_test)
    except Exception as e:
        continue

In [ ]:
if len(oof_train_list) > 0:
    X_oof = np.hstack(oof_train_list)
    X_test_oof = np.hstack(oof_test_list)

    blender = Lasso(alpha=0.0005, random_state=42)
    blender.fit(X_oof, y)
    preds_oof = blender.predict(X_test_oof)
else:
    preds_oof = np.zeros(X_test.shape[0])

In [ ]:
full_preds = []
for name, model in models.items():
    try:
        model.fit(X, y)
        p = model.predict(X_test)
        full_preds.append(p)
    except Exception as e:
        continue

if len(full_preds) > 0:
    avg_full = np.mean(np.vstack(full_preds), axis=0)
    final_preds_log = 0.5 * preds_oof.ravel() + 0.5 * avg_full.ravel()
else:
    final_preds_log = preds_oof.ravel()


final_preds = np.expm1(final_preds_log)


In [ ]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": final_preds
})
submission.to_csv("submission.csv", index=False)
print("✅ submission.csv saved!")